# Exchange Rate (USD/UAH) - Data Cleaning 

### Data Info:
* **Indicator:** `Офіційний курс гривні, грн (usd_uah)`
* **Source:** National Bank of Ukraine: https://bank.gov.ua/ua/markets/exchangerate-chart?cn%5B%5D=USD&startDate=31.12.2014&endDate=25.04.2026 
* **Raw File:** `data_raw/Exchange_Rate.xlsx`
* **Output:** `data_processed/Exchange_Rate_clean.xlsx`  

### Cleaning Notes:
- exchange rate is given per 100 units before 28.12.2019, per 1 unit from 2015 onward 

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path

In [2]:
file_path = Path("../../data_raw/Exchange_Rate.xlsx")
output_path = Path("../../data_processed/Exchange_Rate_clean.xlsx")
output_path.parent.mkdir(parents=True, exist_ok=True)

sheet_name = "Sheet1"

df_raw = pd.read_excel(file_path, sheet_name=sheet_name, header=None)
df_raw

,0,1,2,3,4,5,6,7
0,Дата,Час,Код цифровий,Код літерний,Кількість одиниць,Назва валюти,"Офіційний курс гривні, грн",Примітка
1,31.12.2014,00.00,840,USD,100,Долар США,1576.8556,NaN
2,01.01.2015,00.00,840,USD,100,Долар США,1576.8556,NaN
3,02.01.2015,00.00,840,USD,100,Долар США,1576.8556,NaN
4,03.01.2015,00.00,840,USD,100,Долар США,1576.8556,NaN
...,...,...,...,...,...,...,...,...
4125,16.04.2026,00.00,840,USD,1,Долар США,43.5152,NaN
4126,17.04.2026,00.00,840,USD,1,Долар США,43.6368,NaN
4127,18.04.2026,00.00,840,USD,1,Долар США,43.6368,NaN
4128,19.04.2026,00.00,840,USD,1,Долар США,43.6368,NaN


In [3]:
# set correct header

df = pd.read_excel(file_path, sheet_name=sheet_name, header=0)
display(df.head())
print(df.columns.tolist())

,Дата,Час,Код цифровий,Код літерний,Кількість одиниць,Назва валюти,"Офіційний курс гривні, грн",Примітка
0,31.12.2014,0.0,840,USD,100,Долар США,1576.8556,NaN
1,01.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN
2,02.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN
3,03.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN
4,04.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN


['Дата', 'Час', 'Код цифровий', 'Код літерний', 'Кількість одиниць', 'Назва валюти', 'Офіційний курс гривні, грн', 'Примітка']


In [4]:
# filter needed columns and indicator

date_col = "Дата"
raw_value_col = "Офіційний курс гривні, грн"
units_col = "Кількість одиниць"
indicator_name = "usd_uah"

df["value"] = df[raw_value_col] / df[units_col]

df_filtered = df[[date_col, "value"]].copy()
df_filtered["indicator"] = indicator_name

df

,Дата,Час,Код цифровий,Код літерний,Кількість одиниць,Назва валюти,"Офіційний курс гривні, грн",Примітка,value
0,31.12.2014,0.0,840,USD,100,Долар США,1576.8556,NaN,15.768556
1,01.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN,15.768556
2,02.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN,15.768556
3,03.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN,15.768556
4,04.01.2015,0.0,840,USD,100,Долар США,1576.8556,NaN,15.768556
...,...,...,...,...,...,...,...,...,...
4124,16.04.2026,0.0,840,USD,1,Долар США,43.5152,NaN,43.515200
4125,17.04.2026,0.0,840,USD,1,Долар США,43.6368,NaN,43.636800
4126,18.04.2026,0.0,840,USD,1,Долар США,43.6368,NaN,43.636800
4127,19.04.2026,0.0,840,USD,1,Долар США,43.6368,NaN,43.636800


In [5]:
# clean date strings ??

def clean_date_string(x):
    if pd.isna(x):
        return np.nan
    
    x = str(x).strip()
    
    # remove footnotes like **, #, (A4)
    x = re.sub(r"\*+", "", x)
    x = re.sub(r"#", "", x)
    x = re.sub(r"\s*\(.*?\)", "", x)
    x = x.strip()
    
    return x

df_filtered["date_raw"] = df_filtered[date_col].apply(clean_date_string)
df_filtered[[date_col, "date_raw"]].head(10)

,Дата,date_raw
0,31.12.2014,31.12.2014
1,01.01.2015,01.01.2015
2,02.01.2015,02.01.2015
3,03.01.2015,03.01.2015
4,04.01.2015,04.01.2015
5,05.01.2015,05.01.2015
6,06.01.2015,06.01.2015
7,07.01.2015,07.01.2015
8,08.01.2015,08.01.2015
9,09.01.2015,09.01.2015


In [6]:
# convert dates to datetime

df_filtered["date"] = pd.to_datetime(
    df_filtered["date_raw"],
    dayfirst=True,
    errors="coerce"
)

# fallback for timestamp-like strings if any appear
mask_failed = df_filtered["date"].isna() & df_filtered["date_raw"].notna()
df_filtered.loc[mask_failed, "date"] = pd.to_datetime(
    df_filtered.loc[mask_failed, "date_raw"],
    errors="coerce"
)

display(df_filtered[[date_col, "date_raw", "date"]].head(10))
print("missing dates:", df_filtered["date"].isna().sum())

,Дата,date_raw,date
0,31.12.2014,31.12.2014,2014-12-31
1,01.01.2015,01.01.2015,2015-01-01
2,02.01.2015,02.01.2015,2015-01-02
3,03.01.2015,03.01.2015,2015-01-03
4,04.01.2015,04.01.2015,2015-01-04
5,05.01.2015,05.01.2015,2015-01-05
6,06.01.2015,06.01.2015,2015-01-06
7,07.01.2015,07.01.2015,2015-01-07
8,08.01.2015,08.01.2015,2015-01-08
9,09.01.2015,09.01.2015,2015-01-09


missing dates: 0


In [7]:
df = df_filtered[["date", "value", "indicator"]].copy()
df

,date,value,indicator
0,2014-12-31,15.768556,usd_uah
1,2015-01-01,15.768556,usd_uah
2,2015-01-02,15.768556,usd_uah
3,2015-01-03,15.768556,usd_uah
4,2015-01-04,15.768556,usd_uah
...,...,...,...
4124,2026-04-16,43.515200,usd_uah
4125,2026-04-17,43.636800,usd_uah
4126,2026-04-18,43.636800,usd_uah
4127,2026-04-19,43.636800,usd_uah


In [8]:
# convert values to numeric

df["value"] = (
    df["value"]
    .astype(str)
    .str.replace(r"\s+", "", regex=True)
    .str.replace(",", ".", regex=False)
)

df["value"] = pd.to_numeric(df["value"], errors="coerce")

print("missing values:", df["value"].isna().sum())

missing values: 0


In [9]:
# remove non-data rows if any slipped through

df_clean = df[["date", "value", "indicator"]].copy()

df_clean = df_clean[
    df_clean["date"].notna() &
    df_clean["value"].notna()
].copy()

df_clean = df_clean.sort_values("date").reset_index(drop=True)

df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4129 entries, 0 to 4128
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       4129 non-null   datetime64[ns]
 1   value      4129 non-null   float64       
 2   indicator  4129 non-null   object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 96.9+ KB


In [10]:
# aggregate to monthly frequency (end of month)

df_monthly = (
    df_clean
    .sort_values("date")
    .set_index("date")
    .groupby("indicator")
    .resample("M")["value"]
    .last()   # /.mean() 
    .reset_index()
    [["date", "value", "indicator"]]
)

df_monthly.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137 entries, 0 to 136
Data columns (total 3 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   date       137 non-null    datetime64[ns]
 1   value      137 non-null    float64       
 2   indicator  137 non-null    object        
dtypes: datetime64[ns](1), float64(1), object(1)
memory usage: 3.3+ KB


In [11]:
df_monthly

,date,value,indicator
0,2014-12-31,15.768556,usd_uah
1,2015-01-31,16.157817,usd_uah
2,2015-02-28,27.763120,usd_uah
3,2015-03-31,23.442625,usd_uah
4,2015-04-30,21.046832,usd_uah
...,...,...,...
132,2025-12-31,42.387800,usd_uah
133,2026-01-31,42.848300,usd_uah
134,2026-02-28,43.208100,usd_uah
135,2026-03-31,43.795500,usd_uah


In [12]:
# save

df_monthly["date"] = df_monthly["date"].dt.date
df_monthly.to_excel(output_path, index=False)

print(f"saved to: {output_path}")

saved to: ..\..\data_processed\Exchange_Rate_clean.xlsx
